# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LiquidMercury-tech/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For Lane 4, the unit of analysis is one content item aggregated over a rolling 90-day observation window. The starter dataset is already at page-grain, so each row represents a single page with its 90-day demand, position, CTR, and engagement summary. The time window is the trailing 90-day period captured in the starter file, with content age in days and trend measurements alongside the classic 90-day traffic rates.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
print(f'Rows: {len(df)}')
print(f'Unique clients: {df["client_id"].nunique()}')
print(f'Min content age days: {df["content_age_days"].min()}')
print(f'Max content age days: {df["content_age_days"].max()}')


Rows: 30000
Unique clients: 32
Min content age days: 90
Max content age days: 564


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: `impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `word_count`, `content_type`, `main_intent`, `competition_level`, `content_age_days`, `freshness_tier`, `position_tier`, `trend_pct` (used with caution)

Label / proxy: `low_ctr_opportunity` or `weak_engagement_opportunity` defined from observed signals such as `ctr < 0.5` with high impressions and strong position, or `engagement_rate` below a threshold with enough sessions

Context: `client_id`, `content_id`, `content_type`, `main_intent`, `competition_level`, `age_tier`, `freshness_tier`, `position_tier`, `trend_direction`, `content_age_days`

Excluded: `provider_used`, `model_used`, `keyword_hash_id`-style raw identifiers are excluded because they are metadata or IDs rather than decision features; `trend_direction` is excluded from the feature set when used to define the label, because using the same derived rule both as feature and target creates leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'content_type', 'main_intent', 'competition_level', 'content_age_days', 'freshness_tier', 'position_tier', 'trend_pct']
label_proxy = 'low_ctr_opportunity OR weak_engagement_opportunity'
context = ['client_id', 'content_id', 'content_type', 'main_intent', 'competition_level', 'age_tier', 'freshness_tier', 'position_tier']
excluded = ['provider_used', 'model_used', 'trend_direction']
print(f'Features: {len(features)}+ columns')
print(f'Label proxy: {label_proxy}')
print(f'Excluded: {excluded}')


Features: 15+ columns
Label proxy: low_ctr_opportunity OR weak_engagement_opportunity
Excluded: ['provider_used', 'model_used', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The starter dataset is at page grain, not page-day grain. There are 30,000 rows and 32 unique clients. The position data is not always valid: 1,205 rows have `avg_position = 0`, which means 'no data', not zero ranking. That matters because the actual review queue should only use pages with usable position and enough search demand.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
print(f'Rows: {len(df)}')
print(f'Unique clients: {df["client_id"].nunique()}')
print(f'No-position rows: {int((df["avg_position"] == 0).sum())}')
print(f'Missing CTR: {int(df["ctr"].isna().sum())}')
print(f'Rows with impressions > 0: {int((df["impressions_90d"] > 0).sum())}')
print(f'Visible pages with position > 0: {int((df["avg_position"] > 0).sum())}')


Rows: 30000
Unique clients: 32
No-position rows: 1205
Missing CTR: 0
Rows with impressions > 0: 30000
Visible pages with position > 0: 28795


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot prove causality. It cannot tell us whether improving a title or meta description definitely caused a CTR increase, because there is no randomized experiment or before-and-after intervention in the starter file. It also cannot safely model long-run seasonality or the full client history if a client has sparse early tracking. The main data limits are that the page records are aggregated, not per-day intervention data, and the use of 90-day windows means the model only sees an observable slice of the page lifecycle rather than the full search journey.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
zero_position = int((df['avg_position'] == 0).sum())
print('This project is observational; it cannot prove causality.')
print(f'Position zero rows: {zero_position}')
print('These are excluded from rank-based review scoring.')


This project is observational; it cannot prove causality.
Position zero rows: 1205
These are excluded from rank-based review scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.